In [1]:
import pandas as pd 
import glob
from datetime import datetime 
import os
from bs4 import BeautifulSoup

In [2]:
INPUT_DATA_FOLDER = "<input your path for folder with parsed html>"

In [3]:
files_by_days = glob.glob(f"{INPUT_DATA_FOLDER}/*.html")

In [4]:
len(files_by_days)

print(files_by_days[0])

/Users/mac/Downloads/Python26/data/isw_raw/2023_11_05.html


In [6]:
all_data = []

for file in files_by_days:
    d = {}
    file_name = os.path.basename(file).replace(".html","").split("___")

    date = datetime.strptime(file_name[0], "%Y_%m_%d")
    url = file_name[0]
    
    with open(file,"r") as cfile:
        html_content = cfile.read()
        parsed_html = BeautifulSoup(html_content, 'html.parser')
        
        title = parsed_html.head.find("title").text
        link = parsed_html.head.find("link",attrs={"rel":"canonical"},href=True)
        if link is not None:
            link = link.attrs["href"]
        else:
            link = ""
        text_title = parsed_html.body.find('h1', {'class': 'title', 'id': 'page-title'})
        if text_title is not None:
            text_title = text_title.text
        else:
            text_title = title
        
        d = {
            'id': len(all_data) + 1,
            'date': date.strftime('%Y-%m-%d'),
            'short_url': url,
            'title': title,
            'text_title': text_title,
            'full_url': link,
            'html_content': html_content
        }
         
        all_data.append(d)

In [7]:
data = pd.DataFrame(all_data)

In [8]:
data['html_content'][1]

'<!DOCTYPE html>\n<html lang="en-US">\n<head>\n\t<meta charset="UTF-8">\n\t<meta name=\'robots\' content=\'index, follow, max-image-preview:large, max-snippet:-1, max-video-preview:-1\' />\n<meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover">\n\t<!-- This site is optimized with the Yoast SEO Premium plugin v27.0 (Yoast SEO v27.0) - https://yoast.com/product/yoast-seo-premium-wordpress/ -->\n\t<title>Russian Offensive Campaign Assessment, August 14, 2022 | ISW</title>\n\t<link rel="canonical" href="https://understandingwar.org/research/russia-ukraine/russian-offensive-campaign-assessment_14-25/" />\n\t<meta property="og:locale" content="en_US" />\n\t<meta property="og:type" content="article" />\n\t<meta property="og:title" content="Russian Offensive Campaign Assessment, August 14, 2022" />\n\t<meta property="og:description" content="Russian and proxy troops in Ukraine are likely operating in roughly six groups of forces oriented on Kharkiv City and no

In [9]:
data = data.sort_values(by=['date'])

In [10]:
data

,id,date,short_url,title,text_title,full_url,html_content
1201,1202,2022-02-24,2022_02_24,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n..."
1366,1367,2022-02-25,2022_02_25,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n..."
181,182,2022-02-26,2022_02_26,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n..."
197,198,2022-02-27,2022_02_27,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n..."
1106,1107,2022-02-28,2022_02_28,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n..."
...,...,...,...,...,...,...,...
736,737,2026-02-25,2026_02_25,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n..."
827,828,2026-02-26,2026_02_26,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n..."
1012,1013,2026-02-27,2026_02_27,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n..."
470,471,2026-02-28,2026_02_28,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n..."


In [11]:
from IPython.core.display import HTML, display
display(HTML(data['html_content'][0]))

/var/folders/k6/9kclm4r55wn8573ccdyzgj1m0000gn/T/ipykernel_1495/2979927905.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import HTML, display


1. Remove HTML Content

In [12]:
def remove_names_and_dates(page_html_text):
    parsed_html = BeautifulSoup(page_html_text)
    p_lines = parsed_html.findAll('p')
    
    min_sentense_word_count = 13
    p_index = 0
    
    for p_line in p_lines:
        
        strong_lines = p_line.findAll('strong')
        if not strong_lines:
            continue 
            
        for s in strong_lines:
            if (len(s.text.split(" ")) >= min_sentense_word_count ):
                break
            else:
                p_index += 1
                
                continue
            break
            
        for i in range(0,p_index):
            page_html_text = page_html_text.replace(str(p_lines[i]),"")
            
        return page_html_text

In [13]:
data['html_content_v2'] = data['html_content'].apply(lambda x: remove_names_and_dates(x))

In [14]:
data.head()

,id,date,short_url,title,text_title,full_url,html_content,html_content_v2
1201,1202,2022-02-24,2022_02_24,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n..."
1366,1367,2022-02-25,2022_02_25,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n..."
181,182,2022-02-26,2022_02_26,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n..."
197,198,2022-02-27,2022_02_27,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n..."
1106,1107,2022-02-28,2022_02_28,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n..."


In [15]:
data['html_content_v2'][0]

'<!DOCTYPE html>\n<html lang="en-US">\n<head>\n\t<meta charset="UTF-8">\n\t<meta name=\'robots\' content=\'index, follow, max-image-preview:large, max-snippet:-1, max-video-preview:-1\' />\n<meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover">\n\t<!-- This site is optimized with the Yoast SEO Premium plugin v27.0 (Yoast SEO v27.0) - https://yoast.com/product/yoast-seo-premium-wordpress/ -->\n\t<title>Russian Offensive Campaign Assessment, November 5, 2022 | ISW</title>\n\t<link rel="canonical" href="https://understandingwar.org/research/russia-ukraine/russian-offensive-campaign-assessment_5-24/" />\n\t<meta property="og:locale" content="en_US" />\n\t<meta property="og:type" content="article" />\n\t<meta property="og:title" content="Russian Offensive Campaign Assessment, November 5, 2022" />\n\t<meta property="og:description" content="Wagner Group financier Yevgeny Prigozhin seeks to obfuscate his efforts to strengthen his independent power base with a

2. Remove all numbers

In [17]:
import re

pattern = "\[(\d+)\]"


data['html_content_v3'] = data['html_content_v2'].apply(lambda x: re.sub(pattern,'',str(x)))

In [18]:
data['html_content_v3'][0]

'<!DOCTYPE html>\n<html lang="en-US">\n<head>\n\t<meta charset="UTF-8">\n\t<meta name=\'robots\' content=\'index, follow, max-image-preview:large, max-snippet:-1, max-video-preview:-1\' />\n<meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover">\n\t<!-- This site is optimized with the Yoast SEO Premium plugin v27.0 (Yoast SEO v27.0) - https://yoast.com/product/yoast-seo-premium-wordpress/ -->\n\t<title>Russian Offensive Campaign Assessment, November 5, 2022 | ISW</title>\n\t<link rel="canonical" href="https://understandingwar.org/research/russia-ukraine/russian-offensive-campaign-assessment_5-24/" />\n\t<meta property="og:locale" content="en_US" />\n\t<meta property="og:type" content="article" />\n\t<meta property="og:title" content="Russian Offensive Campaign Assessment, November 5, 2022" />\n\t<meta property="og:description" content="Wagner Group financier Yevgeny Prigozhin seeks to obfuscate his efforts to strengthen his independent power base with a

3.Remove list of references

In [19]:
data['html_content_v4'] = data['html_content_v3'].apply(lambda x: BeautifulSoup(x).text)

In [20]:
data['html_content_v4'][0]

"\n\n\n\n\n\n\nRussian Offensive Campaign Assessment, November 5, 2022 | ISW\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n \n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nOptions\n\n\n\t\t\t\tDOWNLOAD PAGE\n\t\t\t  \nPRINT PAGE\n\nShare\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nSkip to contentSkip to Content \n\n\n\n\n\n \n \n\n\nDonate \nMenu \n\n\n\n\n\n\n\n \nMenu \nAbout ISW\n\n\n\nAbout ISW\n\n\n\nOur Capabilities\nISW’s Comparative Advantage\n\n\nOur History\nImproving the National Security Debate\n\n\n\n\nWho We Are\nMeet the Team\n\n\nCareers\nJoin the Mission\n\n\nOpen Positions\n\n\n\n\n\n\n\nMAP ROOM\nISW produces the world’s premier open-source conflict maps\n  SEE MAPS\n\n\n\n\n\n\n\n\nAnalysis\n\n\n\nAnalysis\nThe General Jack Keane Center for National Security\n\n\n\nMap Room\nDive into ISW's Complete Map Catalog\n\n\nBriefing Room\nVideos, Podcasts, and Interactive Content\n\n\nResearch Library\nBrowse ISW's Entire 

In [21]:
data.head(5)

,id,date,short_url,title,text_title,full_url,html_content,html_content_v2,html_content_v3,html_content_v4
1201,1202,2022-02-24,2022_02_24,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...",\n\n\n\n\n\n\nRussian Offensive Campaign Asses...
1366,1367,2022-02-25,2022_02_25,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...",\n\n\n\n\n\n\nRussian Offensive Campaign Asses...
181,182,2022-02-26,2022_02_26,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...",\n\n\n\n\n\n\nRussian Offensive Campaign Asses...
197,198,2022-02-27,2022_02_27,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...",\n\n\n\n\n\n\nRussian Offensive Campaign Asses...
1106,1107,2022-02-28,2022_02_28,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...",\n\n\n\n\n\n\nRussian Offensive Campaign Asses...


In [22]:
data['html_content_v5'] = data['html_content_v4'].apply(lambda x: re.sub(r'http(\S+.*\s)',"",x))
data['html_content_v6'] = data['html_content_v5'].apply(lambda x: re.sub(r'2022|2023|©2022|©2023|\xa0|\n',"",x))

In [23]:
data['html_content_v6'][0]

"Russian Offensive Campaign Assessment, November 5,  | ISW Options\t\t\t\tDOWNLOAD PAGE\t\t\t  PRINT PAGEShareSkip to contentSkip to Content   Donate Menu  Menu About ISWAbout ISWOur CapabilitiesISW’s Comparative AdvantageOur HistoryImproving the National Security DebateWho We AreMeet the TeamCareersJoin the MissionOpen PositionsMAP ROOMISW produces the world’s premier open-source conflict maps  SEE MAPSAnalysisAnalysisThe General Jack Keane Center for National SecurityMap RoomDive into ISW's Complete Map CatalogBriefing RoomVideos, Podcasts, and Interactive ContentResearch LibraryBrowse ISW's Entire Body of WorkTEAMS/PORTFOLIOSRussia & UkraineMiddle EastChina & TaiwanDefense of EuropeAdversary EntenteContemporary & Future WarGeospatial IntelligenceCognitive WarfareEducationEducationThe General David H. Petraeus Center for Emerging LeadersHertog War StudiesISW’s Premier Educational Program for UndergraduatesFellowshipsEnrichment and Advancement Opportunities for Entry-Level and Mid-Car

In [24]:
data.head()

,id,date,short_url,title,text_title,full_url,html_content,html_content_v2,html_content_v3,html_content_v4,html_content_v5,html_content_v6
1201,1202,2022-02-24,2022_02_24,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...",\n\n\n\n\n\n\nRussian Offensive Campaign Asses...,\n\n\n\n\n\n\nRussian Offensive Campaign Asses...,"Russian Offensive Campaign Assessment, Februar..."
1366,1367,2022-02-25,2022_02_25,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...",\n\n\n\n\n\n\nRussian Offensive Campaign Asses...,\n\n\n\n\n\n\nRussian Offensive Campaign Asses...,"Russian Offensive Campaign Assessment, Februar..."
181,182,2022-02-26,2022_02_26,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...",\n\n\n\n\n\n\nRussian Offensive Campaign Asses...,\n\n\n\n\n\n\nRussian Offensive Campaign Asses...,"Russian Offensive Campaign Assessment, Februar..."
197,198,2022-02-27,2022_02_27,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...",\n\n\n\n\n\n\nRussian Offensive Campaign Asses...,\n\n\n\n\n\n\nRussian Offensive Campaign Asses...,"Russian Offensive Campaign Assessment, Februar..."
1106,1107,2022-02-28,2022_02_28,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...",\n\n\n\n\n\n\nRussian Offensive Campaign Asses...,\n\n\n\n\n\n\nRussian Offensive Campaign Asses...,"Russian Offensive Campaign Assessment, Februar..."


In [25]:
data2 = data.drop(['html_content_v2','html_content_v3','html_content_v4','html_content_v5'],axis=1)

In [26]:
data2.head()

,id,date,short_url,title,text_title,full_url,html_content,html_content_v6
1201,1202,2022-02-24,2022_02_24,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","Russian Offensive Campaign Assessment, Februar..."
1366,1367,2022-02-25,2022_02_25,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","Russian Offensive Campaign Assessment, Februar..."
181,182,2022-02-26,2022_02_26,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","Russian Offensive Campaign Assessment, Februar..."
197,198,2022-02-27,2022_02_27,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","Russian Offensive Campaign Assessment, Februar..."
1106,1107,2022-02-28,2022_02_28,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","Russian Offensive Campaign Assessment, Februar..."


In [27]:
#data2.to_csv('all_isw_reports_full.csv')